## Problem: Write a Transformer

### Problem Statement
Implement a **Transformer model** in PyTorch by completing the required sections. The model should consist of an embedding layer, a Transformer encoder, and an output layer for sequence processing and prediction.

### Requirements
1. **Define the Transformer Model Architecture**:
   - **Embedding Layer**:
     - Implement a layer to transform input data into a higher-dimensional space.
     - Use a `torch.nn.Linear` or `torch.nn.Embedding` layer to create embeddings from the input.
   - **Transformer Encoder**:
     - Use `torch.nn.TransformerEncoder` or `torch.nn.Transformer` to process sequences with attention.
     - Configure parameters such as the number of attention heads and encoder layers.
   - **Output Layer**:
     - Add a fully connected (linear) layer to reduce the transformer's sequence output into the desired output dimension.

2. **Implement the Forward Method**:
   - Map the input to the higher-dimensional space using the embedding layer.
   - Pass the transformed input through the Transformer encoder.
   - Use the output layer to convert the encoded sequence into predictions.

### Constraints
- Handle input padding correctly for variable-length sequences.
- Ensure compatibility with batch processing by correctly shaping input and output tensors.


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [60]:
def scaled_dot_product_attention(query, key, value, attn_mask=None, is_causal=False):
    #     query: B, H, L, D
    #       key: B, H, S, D
    #     value: B, H, S, D
    # attn_mask: B, H, L, S
    
    B, H, L, D = query.shape
    _, _, S, _ = key.shape
    
    device = query.device
    
    attn_logits = torch.einsum("bhld,bhsd->bhls", query, key) * (1/D)**0.5

    attn_bias = torch.zeros((B, H, L, S), device=device)
    if attn_mask is not None:
        assert not is_causal, "when attn_mask is specified, is_causal can not be used"
        attn_bias.masked_fill_(~attn_mask, -torch.inf)
    if is_causal:
        assert attn_mask is None, "when is_casual is specified, attn_mask can not be used"
        assert L == S, "causal not defined when seq length and kv length differs"
        temp_mask = torch.tril(torch.ones((L, S), dtype=torch.bool), diagonal=0)
        attn_bias.masked_fill_(~temp_mask, -torch.inf)

    attn_weight = F.softmax(attn_logits + attn_bias, dim=-1)
    
    output = torch.einsum("bhls,bhsd->bhld", attn_weight, value)
    return output

In [61]:
import torch, math, torch.nn.functional as F

def ref(q, k, v, attn_mask=None, is_causal=False):
    # PyTorch ≥2.0: F.scaled_dot_product_attention
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=is_causal)

torch.manual_seed(0)
B,H,L,S,D = 2, 3, 5, 5, 8
q = torch.randn(B,H,L,D)
k = torch.randn(B,H,S,D)
v = torch.randn(B,H,S,D)

mask = torch.randint(0,2,(B,H,L,S)).bool()
is_causal=False

out_user   = scaled_dot_product_attention(q,k,v, attn_mask=mask, is_causal=is_causal)
out_pytorch= ref(q,k,v, attn_mask=mask, is_causal=is_causal)
assert torch.allclose(out_user, out_pytorch, atol=1e-5), "mismatch!"

In [31]:
# Define a Transformer Model
#TODO: Implement a Transformer model
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        
        self.query_projection = nn.Linear(embed_dim, embed_dim)
        self.key_projection = nn.Linear(embed_dim, embed_dim)
        self.value_projection = nn.Linear(embed_dim, embed_dim)

        self.output_projection = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # x: B, L, D

        B, L, D = x.shape
        H = self.num_heads
        
        query = self.query_projection(x)
        key = self.key_projection(x)
        value = self.value_projection(x)

        query = query.view(B, L, H, D//H).permute(0, 2, 1, 3)
        key = key.view(B, L, H, D//H).permute(0, 2, 1, 3)
        value = value.view(B, L, H, D//H).permute(0, 2, 1, 3) # B, H, L, D//H

        output = scaled_dot_product_attention(query, key, value) # B, H, L, D//H
        output = self.output_projection(output.permute(0, 2, 1, 3).reshape(B, L, D))
        return output


class TransformerLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super(TransformerLayer, self).__init__()
        assert embed_dim % num_heads == 0
        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, ff_dim), nn.ReLU(), nn.Linear(ff_dim, embed_dim))
        self.layer_norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attention(x)
        x = self.layer_norm1(x)
        x = x + self.ffn(x)
        x = self.layer_norm2(x)
        return x


class TransformerModel(nn.Module):
    def __init__(self, input_dim, embed_dim, num_heads, num_layers, ff_dim, output_dim):
        super(TransformerModel, self).__init__()
        # Add layers and modules for embedding, transformer, and output
        self.input_projection = nn.Linear(input_dim, embed_dim)
        self.transformer_layers = nn.Sequential(*[TransformerLayer(embed_dim, num_heads, ff_dim) for _ in range(num_layers)])
        self.output_projection = nn.Linear(embed_dim, output_dim)

    def forward(self, x):
        # Define the forward pass logic
        x = self.input_projection(x)
        x = self.transformer_layers(x)
        x = self.output_projection(x)
        return x

In [32]:
# Generate synthetic data
torch.manual_seed(42)
seq_length = 10
num_samples = 100
input_dim = 1
X = torch.rand(num_samples, seq_length, input_dim)  # Random sequences
y = torch.sum(X, dim=1)  # Target is the sum of each sequence

# Initialize the model, loss function, and optimizer
input_dim = 1
embed_dim = 16
num_heads = 2
num_layers = 2
ff_dim = 64
output_dim = 1

model = TransformerModel(input_dim, embed_dim, num_heads, num_layers, ff_dim, output_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [33]:
# Training loop
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    predictions = model(X)[:, -1, :]
    loss = criterion(predictions, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [100/1000], Loss: 2.3874
Epoch [200/1000], Loss: 0.9176
Epoch [300/1000], Loss: 0.8755
Epoch [400/1000], Loss: 0.4158
Epoch [500/1000], Loss: 0.0716
Epoch [600/1000], Loss: 0.0340
Epoch [700/1000], Loss: 0.0210
Epoch [800/1000], Loss: 0.0136
Epoch [900/1000], Loss: 0.0097
Epoch [1000/1000], Loss: 0.0069


In [41]:
# Testing on new data
X_test = torch.rand(2, seq_length, input_dim)
y_test = torch.sum(X_test, dim=1)
with torch.no_grad():
    predictions = model(X_test)[:,-1,:]
    print(f"Predictions for {X_test.tolist()}: {predictions.tolist()}")

Predictions for [[[0.3758213520050049], [0.14101773500442505], [0.3692624568939209], [0.05245780944824219], [0.9324256777763367], [0.7140067219734192], [0.6312100887298584], [0.1618126630783081], [0.17431598901748657], [0.7851710915565491]], [[0.09490090608596802], [0.5969802737236023], [0.8740814924240112], [0.06404381990432739], [0.8529677391052246], [0.08781552314758301], [0.4028906226158142], [0.8452677726745605], [0.8707805275917053], [0.9540941715240479]]]: [[4.325301647186279], [5.629753589630127]]


In [42]:
y_test

tensor([[4.3375],
        [5.6438]])